# 01 Data preparation

This notebook turns the raw KinNews CSVs into the deduplicated train, development, and test splits that every later notebook reads.

It needs the raw KINNEWS release in `data/source/KINNEWS/raw/` (see `data/README.md`). Its output is committed in `data/raw/`, so the other notebooks run without the CSVs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/01_data_preparation.ipynb)

In [1]:
# Setup: run from the repository root, on a laptop or on Colab.
import os
import subprocess
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/Ikinyamakuru").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Keaane/Ikinyamakuru.git", "/content/Ikinyamakuru"], check=True)
    os.chdir("/content/Ikinyamakuru")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path("src").resolve()))

# Show each warning as one short line.
warnings.showwarning = lambda message, category, *args, **kwargs: print(f"{category.__name__}: {message}")

In [2]:
import csv
import json
from collections import Counter, defaultdict

import numpy as np
from sklearn.model_selection import train_test_split

from inkuru.data import canonical, normalize, write_jsonl
from inkuru.labels import LABELS
from inkuru.paths import DATA, DEV_FRACTION, RAW, RESULTS, SEED, SOURCE

## Read the raw files

Each row has a class index, an English and a Kinyarwanda section name, a URL, a title, and a body. The CSV calls class 1 `politic`, which becomes `politics`, and every row's index is checked against its section name. The model text is the title, a line break, and the body, with stopwords and punctuation kept because the apostrophe is part of Kinyarwanda spelling.

In [3]:
def read_csv(path) -> list[dict]:
    csv.field_size_limit(10**8)  # some article bodies are longer than the default limit
    rows = []
    with path.open(newline="", encoding="utf-8") as handle:
        for index, row in enumerate(csv.DictReader(handle)):
            raw_label = (row.get("en_label") or "").strip().lower()
            label = canonical(raw_label)
            number = (row.get("label") or "").strip()
            title = (row.get("title") or "").replace("\x00", " ").strip()
            content = (row.get("content") or "").replace("\x00", " ").strip()
            kin = (row.get("kin_label") or "").strip().lower()
            if label not in LABELS:
                raise ValueError(f"{path.name} row {index}: unknown section {raw_label!r}")
            expected = LABELS[int(number) - 1]
            if expected != label:
                raise ValueError(f"{path.name} row {index}: label {number} is {expected}, file says {label}")
            text = f"{title}\n{content}".strip()
            if len(normalize(text)) < 20:
                continue
            rows.append({
                "source_row": index,
                "label": label,
                "kin_label": kin,
                "title": title,
                "content": content,
                "text": text,
                "url": (row.get("url") or "").strip(),
            })
    return rows


if not (SOURCE / "train.csv").exists():
    raise FileNotFoundError("Put the raw KinNews train.csv and test.csv in data/source/KINNEWS/raw/ (see data/README.md).")
train_all = read_csv(SOURCE / "train.csv")
test_all = read_csv(SOURCE / "test.csv")
print(f"training rows {len(train_all):,}, test rows {len(test_all):,}")

training rows 17,014, test rows 4,254


## Repeated articles

Many articles appear more than once in the official training file, some with different section labels across their copies. The key is the lowercased text with whitespace collapsed.

In [4]:
copies = Counter(normalize(row["text"]) for row in train_all)
labels_per_text = defaultdict(set)
for row in train_all:
    labels_per_text[normalize(row["text"])].add(row["label"])
conflicts = Counter(" / ".join(sorted(labels)) for labels in labels_per_text.values() if len(labels) > 1)

print(f"distinct training texts {len(copies):,} out of {len(train_all):,} rows")
print(f"most copies of one text: {max(copies.values())}")
print(f"repeated texts with more than one label: {sum(conflicts.values())}")
for pair, count in conflicts.most_common(5):
    print(f"   {pair}: {count}")

distinct training texts 7,630 out of 17,014 rows
most copies of one text: 80
repeated texts with more than one label: 156
   entertainment / sport: 97
   politics / sport: 27
   economy / politics: 8
   health / politics: 6
   entertainment / fashion / health / relationship / sport: 4


## Remove the duplicates and the train and test overlap

The first copy of each article is kept. Training articles that also appear in the test file are dropped, so no story is both trained on and scored.

In [5]:
dropped = Counter()


def dedupe(rows: list[dict], split: str) -> list[dict]:
    seen: set[str] = set()
    kept = []
    for row in rows:
        key = normalize(row["text"])
        if key in seen:
            dropped[f"duplicate_{split}"] += 1
            continue
        seen.add(key)
        kept.append(dict(row))
    return kept


train_unique = dedupe(train_all, "train_file")
test_rows = dedupe(test_all, "test_file")
for row in test_rows:
    row["id"] = f"test-{row['source_row']:05d}"
    row["split"] = "test"

test_keys = {normalize(row["text"]) for row in test_rows}
filtered = []
for row in train_unique:
    if normalize(row["text"]) in test_keys:
        dropped["overlaps_test"] += 1
        continue
    filtered.append(row)
train_unique = filtered
print(dict(dropped))

{'duplicate_train_file': 9384, 'duplicate_test_file': 790, 'overlaps_test': 3}


## Development split

KinNews has no validation split, so a stratified 10 percent of the remaining training articles becomes development, with seed 42. The test file stays the official test file.

In [6]:
labels = [row["label"] for row in train_unique]
train_idx, dev_idx = train_test_split(
    np.arange(len(train_unique)), test_size=DEV_FRACTION, random_state=SEED, stratify=labels
)
train_rows, dev_rows = [], []
for index in sorted(int(i) for i in train_idx):
    item = dict(train_unique[index])
    item["id"] = f"train-{item['source_row']:05d}"
    item["split"] = "train"
    train_rows.append(item)
for index in sorted(int(i) for i in dev_idx):
    item = dict(train_unique[index])
    item["id"] = f"dev-{item['source_row']:05d}"
    item["split"] = "dev"
    dev_rows.append(item)
print(f"train {len(train_rows):,}, development {len(dev_rows):,}, test {len(test_rows):,}")

train 6,864, development 763, test 3,464


## Write the splits and the data profile

In [7]:
RAW.mkdir(parents=True, exist_ok=True)
write_jsonl(RAW / "train.jsonl", train_rows)
write_jsonl(RAW / "dev.jsonl", dev_rows)
write_jsonl(RAW / "test.jsonl", test_rows)

kin_counts = defaultdict(Counter)
for row in train_rows + dev_rows + test_rows:
    if row["kin_label"]:
        kin_counts[row["label"]][row["kin_label"]] += 1
kin_map = {label: (kin_counts[label].most_common(1)[0][0] if kin_counts[label] else label) for label in LABELS}
(DATA / "kin_labels.json").write_text(json.dumps(kin_map, ensure_ascii=False, indent=2) + "\n")


def pack(rows: list[dict]) -> dict:
    counts = Counter(row["label"] for row in rows)
    words = [len(row["text"].split()) for row in rows]
    return {
        "n": len(rows),
        "counts": {label: counts[label] for label in LABELS},
        "words_mean": float(np.mean(words)) if words else 0.0,
        "words_median": float(np.median(words)) if words else 0.0,
    }


profile = {
    "dataset": "KinNews raw",
    "source": "Niyongabo et al., 2020. Official train/test kept. Dev is 10% of train, seed 42, stratified.",
    "dropped": dict(dropped),
    "kin_labels": kin_map,
    "train": pack(train_rows),
    "dev": pack(dev_rows),
    "test": pack(test_rows),
    "non_ascii_share_train": float(np.mean([
        any(ord(ch) > 127 for ch in row["text"]) for row in train_rows
    ])),
}
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "data_profile.json").write_text(json.dumps(profile, indent=2) + "\n")

print(f"{'section':15s}{'train':>8s}{'dev':>6s}{'test':>7s}")
for label in LABELS:
    print(f"{label:15s}{profile['train']['counts'][label]:8,d}{profile['dev']['counts'][label]:6,d}{profile['test']['counts'][label]:7,d}")
print(f"{'total':15s}{profile['train']['n']:8,d}{profile['dev']['n']:6,d}{profile['test']['n']:7,d}")
print(f"\nmean words: train {profile['train']['words_mean']:.0f}, test {profile['test']['words_mean']:.0f}")

section           train   dev   test
politics          1,628   181    315
sport               878    98    744
economy           1,000   111    174
health              978   109    272
entertainment       994   110    774
history             196    22     11
technology          136    15    103
tourism              31     3     76
culture              69     8    138
fashion              34     4     91
religion            401    44    256
environment          34     4    100
education           104    12     70
relationship        381    42    340
total             6,864   763  3,464

mean words: train 347, test 354
